# Planar Predictive Coding — persistent hidden states

**Architecture:** `X (2,m) → H1 (6,m) → H2 (4,m) → p (1,m)`.

**Key change:** create `H1` and `H2` **once for the whole training dataset**, before the outer training loop. Every training sample still has its own column in those matrices. Keep the columns alive across all weight updates, instead of replacing them with a new forward pass on every iteration.

Training alternates between (a) adjusting the same hidden activities while keeping weights fixed and (b) updating weights from the resulting local errors. This is a **persistent-state variant** of the earlier PC experiment. As weights evolve, the stored activities become imperfect guesses; inference adjusts them again. Persistence is an implementation choice, not a requirement of all PC models.

**Run from the folder containing `planar_utils.py`** (the same utility your original notebook imports).

**Steps:** import data → define layers → initialize weights → forward propagation → initialize persistent states once → calculate prediction errors / energy → iterative inference → weight gradients and update → integrate training.

## 1. Import libraries and dataset — **unchanged**

In [1]:
import numpy as np
from planar_utils import load_planar_dataset, sigmoid

X, Y = load_planar_dataset()
print("X:", X.shape, "Y:", Y.shape)  # (2, 400), (1, 400)

X: (2, 400) Y: (1, 400)


## 2. Define layers — **unchanged**

In [2]:
def layers(X, Y):
    n_x = X.shape[0]
    h1 = 6
    h2 = 4
    n_y = Y.shape[0]
    return (n_x, h1, h2, n_y)

## 3. Initialize weights and biases — **unchanged**

We initialize the weights once. `W1`/`W2` use small random initialization, while `W3` uses a larger scale for the output signal. Hidden states are **different from weights**: they get their own one-time initialization later, using the forward pass on the full training dataset.

In [3]:
def initialize(n_x, h1, h2, n_y):
    W1 = np.random.rand(h1, n_x) * 0.01
    b1 = np.zeros((h1, 1))
    W2 = np.random.rand(h2, h1) * 0.01
    b2 = np.zeros((h2, 1))
    W3 = np.random.rand(n_y, h2)  # slightly larger output weights help signal reach hidden layers
    b3 = np.zeros((n_y, 1))

    return {"W1": W1, "W2": W2, "W3": W3,
            "b1": b1, "b2": b2, "b3": b3}

## 4. Forward propagation — **same equations as backprop**

We will call this function **once before training** to get the initial values of the persistent `H1` and `H2`. Later calls to forward propagation are only for **evaluation** (ordinary prediction from `X`) and do **not** overwrite the persistent hidden states.

Each hidden state matrix has one column per training example: `H1.shape == (6, m)` and `H2.shape == (4, m)`. These are not single shared vectors for all examples.

In [4]:
def forwared_propagation(X, parameters):
    W1, W2, W3 = parameters["W1"], parameters["W2"], parameters["W3"]
    b1, b2, b3 = parameters["b1"], parameters["b2"], parameters["b3"]

    Z1 = W1 @ X + b1
    A1 = np.tanh(Z1)
    Z2 = W2 @ A1 + b2
    A2 = np.tanh(Z2)
    Z3 = W3 @ A2 + b3
    A3 = sigmoid(Z3)

    return {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2,
            "Z3": Z3, "A3": A3}

## 5a. Calculate prediction errors — **new**

For each layer, **error = current activity − predicted activity**:

$$
\mu_1=\tanh(W_1X+b_1),\quad e_1=H_1-\mu_1;
\qquad \mu_2=\tanh(W_2H_1+b_2),\quad e_2=H_2-\mu_2.
$$

For the output, $p=\sigma(W_3H_2+b_3)$ and $dZ_3=p-Y$ is the **BCE derivative with respect to its logit**, not another hidden squared-error residual. Notice `mu2` depends on **current `H1`**, not the old forward `A1`.

In [5]:
def compute_errors(X, Y, parameters, H1, H2):
    mu1 = np.tanh(parameters["W1"] @ X + parameters["b1"])
    mu2 = np.tanh(parameters["W2"] @ H1 + parameters["b2"])
    Z3 = parameters["W3"] @ H2 + parameters["b3"]
    p = sigmoid(Z3)

    e1 = H1 - mu1
    e2 = H2 - mu2
    dZ3 = p - Y

    return {"mu1": mu1, "mu2": mu2, "Z3": Z3, "p": p,
            "e1": e1, "e2": e2, "dZ3": dZ3}

## 5b. Calculate energy — **one scalar assembled from the errors**

$$
E=\frac1m\left[\frac12\sum e_1^2+\frac12\sum e_2^2+
\sum_j\underbrace{\big(\log(1+e^{Z_{3,j}})-Y_jZ_{3,j}\big)}_{\mathrm{BCE\ from\ logits}}\right].
$$

`compute_cost(Y, cache)` measures ordinary forward BCE, exactly as in your backprop workflow. `compute_energy` is mainly useful for understanding and checking whether inference is improving the objective. `compute_errors` gives the *local signals* used in the derivatives. Because we start `H1, H2` from the forward pass, both hidden errors initially equal zero.

In [6]:
def compute_cost(Y, cache):
    # Ordinary forward BCE: use for evaluating predictions.
    return np.mean(np.logaddexp(0, cache["Z3"]) - Y * cache["Z3"])


def compute_energy(Y, errors):
    # PC energy: use to understand hidden-state inference and weight gradients.
    hidden_loss = 0.5 * np.sum(errors["e1"]**2) + 0.5 * np.sum(errors["e2"]**2)
    output_loss = np.sum(np.logaddexp(0, errors["Z3"]) - Y * errors["Z3"])
    m = Y.shape[1]
    return (hidden_loss + output_loss) / m

## 5c. Infer hidden states — **update the states passed in**

For each sample, the activity gradients of the PC energy are

$$
g_{H_2}= e_2+W_3^T(p-Y), \qquad
 g_{H_1}=e_1-W_2^T\big(e_2\odot(1-\mu_2^2)\big).
$$

They are gradients of the **per-sample** energy, so we do not divide them by the number of examples. Unlike the first notebook, **`inference` no longer initializes H1/H2 internally** and has no `cache` argument. It receives the stored arrays and updates them **in place**. Weights stay fixed during each inner loop.

With *initial* forward activities, the first inference step moves `H2` before `H1`. In later outer iterations, errors may already be nonzero because the activities persist.

In [7]:
def inference(X, Y, parameters, H1, H2, steps=20, activity_rate=0.1):
    # H1 and H2 already exist. DO NOT initialize them again here.
    for i in range(steps):
        errors = compute_errors(X, Y, parameters, H1, H2)

        # d(energy of one sample)/d(hidden activities)
        dH2 = errors["e2"] + parameters["W3"].T @ errors["dZ3"]
        dH1 = errors["e1"] - parameters["W2"].T @ (errors["e2"] * (1 - errors["mu2"]**2))

        # Same persistent arrays, updated in place (not reset from A1/A2).
        H1 -= activity_rate * dH1
        H2 -= activity_rate * dH2

    errors = compute_errors(X, Y, parameters, H1, H2)
    return H1, H2, errors

## 6a. Calculate local weight gradients — **new way to get `dW` and `db`**

Now hold the inferred `H1` and `H2` fixed. Each layer adjusts its **weights** to better predict its current target activity:

$$
\begin{aligned}
 dZ_1&=-e_1\odot(1-\mu_1^2), &dW_1&=dZ_1X^T/m,\\
 dZ_2&=-e_2\odot(1-\mu_2^2), &dW_2&=dZ_2H_1^T/m,\\
 dZ_3&=p-Y, &dW_3&=dZ_3H_2^T/m.
\end{aligned}
$$

The biases use `np.sum(dZi, axis=1, keepdims=True) / m`. The minus signs at layers 1 and 2 follow from `e = activity - prediction`. Matrix multiplication uses `@` (equivalent to `np.dot` for these two-dimensional arrays). We divide by `m` because the **weight objective is averaged across samples**.

In [8]:
def parameter_gradients(X, Y, parameters, H1, H2, errors):
    m = X.shape[1]

    dZ1 = -errors["e1"] * (1 - errors["mu1"]**2)
    dZ2 = -errors["e2"] * (1 - errors["mu2"]**2)
    dZ3 = errors["dZ3"]

    dW1 = (dZ1 @ X.T) / m
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m
    dW2 = (dZ2 @ H1.T) / m
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m
    dW3 = (dZ3 @ H2.T) / m
    db3 = np.sum(dZ3, axis=1, keepdims=True) / m

    return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2,
            "dW3": dW3, "db3": db3}

## 6b. Update weights — **same subtraction as backpropagation**

In [9]:
def weight_update(parameters, grads, weight_rate=0.1):
    for name in ("W1", "b1", "W2", "b2", "W3", "b3"):
        parameters[name] = parameters[name] - weight_rate * grads["d" + name]
    return parameters

## 7. Integrate training — **initialize activities ONCE**

1. Initialize weights and run forward propagation on all `m` training examples **one time**.
2. Initialize `H1` (shape `h1 × m`) and `H2` (shape `h2 × m`) by copying that initial forward result **one time**.
3. Each outer iteration: refine those **same** states with `inference`, then compute local weight gradients and update weights **once**.
4. Optionally run fresh forward passes **only to evaluate accuracy / ordinary BCE**, without changing `H1` or `H2`.

`H1[:, j]` belongs to sample `X[:, j]`; the array contains different states for different samples. The returned `states` dictionary lets you inspect the persisted values after training. This is full-batch training: if you later use mini-batches, you must retrieve and write back the correct columns by sample index.

In [10]:
def predict(X, parameters):
    cache = forwared_propagation(X, parameters)
    return (cache["A3"] >= 0.5).astype(int)


def pc_model(X, Y, h1=6, h2=4, iterations=10000,
             steps=20, activity_rate=0.1, weight_rate=0.1):
    n_x, _, _, n_y = layers(X, Y)
    parameters = initialize(n_x, h1, h2, n_y)
    history = []

    # ONE-TIME hidden-state initialization for the entire training dataset.
    first_cache = forwared_propagation(X, parameters)
    H1 = first_cache["A1"].copy()  # (h1, m), one column per example
    H2 = first_cache["A2"].copy()  # (h2, m), one column per example

    for i in range(iterations + 1):
        # Evaluation forward pass does not reset/change H1 or H2.
        if i % 1000 == 0 or i == iterations:
            cache = forwared_propagation(X, parameters)
            forward_bce = compute_cost(Y, cache)
            accuracy = np.mean((cache["A3"] >= 0.5) == Y)
            history.append((i, float(forward_bce), float(accuracy)))
            print(f"Iteration {i:5d}: BCE = {forward_bce:.5f}, accuracy = {accuracy:.2%}")

        if i == iterations:
            break

        # Reuse and refine the SAME activities across outer iterations.
        H1, H2, errors = inference(X, Y, parameters, H1, H2, steps, activity_rate)
        grads = parameter_gradients(X, Y, parameters, H1, H2, errors)
        parameters = weight_update(parameters, grads, weight_rate)

    return parameters, history, {"H1": H1, "H2": H2}

## Run the model

`states["H1"]` and `states["H2"]` contain the persistent, trained hidden activities for the 400 training examples. Forward predictions still use the network weights and `X` only, as they must for unseen examples without known labels.

Persistent-state training can behave differently from restarting inference from a forward pass every time. In particular, hidden states adapt to old weights, so inspect forward BCE and accuracy as well as PC energy.

In [11]:
np.random.seed(3)
parameters, history, states = pc_model(
    X, Y, h1=6, h2=4, iterations=10000,
    steps=20, activity_rate=0.1, weight_rate=0.1
)
print("Persistent H1:", states["H1"].shape)
print("Persistent H2:", states["H2"].shape)

Iteration     0: BCE = 0.69320, accuracy = 40.00%


/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_1378/364709847.py:7: RuntimeWarning: divide by zero encountered in matmul
  Z2 = W2 @ A1 + b2
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_1378/364709847.py:7: RuntimeWarning: overflow encountered in matmul
  Z2 = W2 @ A1 + b2
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_1378/364709847.py:7: RuntimeWarning: invalid value encountered in matmul
  Z2 = W2 @ A1 + b2
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_1378/364709847.py:9: RuntimeWarning: divide by zero encountered in matmul
  Z3 = W3 @ A2 + b3
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_1378/364709847.py:9: RuntimeWarning: overflow encountered in matmul
  Z3 = W3 @ A2 + b3
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_1378/364709847.py:9: RuntimeWarning: invalid value encountered in matmul
  Z3 = W3 @ A2 + b3
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_1378/467788396.py:3: RuntimeWarning: 

Iteration  1000: BCE = 0.72483, accuracy = 46.75%
Iteration  2000: BCE = 0.74311, accuracy = 46.75%
Iteration  3000: BCE = 0.65710, accuracy = 56.00%
Iteration  4000: BCE = 0.54783, accuracy = 80.00%
Iteration  5000: BCE = 0.47401, accuracy = 83.25%
Iteration  6000: BCE = 0.44608, accuracy = 83.75%
Iteration  7000: BCE = 0.43102, accuracy = 83.75%
Iteration  8000: BCE = 0.42796, accuracy = 84.50%
Iteration  9000: BCE = 0.43302, accuracy = 83.50%
Iteration 10000: BCE = 0.44968, accuracy = 82.75%
Persistent H1: (6, 400)
Persistent H2: (4, 400)


## Optional sanity check — inference continues from the **stored** activities

Take copies of the already-trained persistent states **only for this diagnostic**, so the test does not overwrite `states`. Fix the trained weights, perform a few more inference steps, and check whether energy decreases. **No new forward initialization** is used for these diagnostic activities.

In [12]:
H1_check = states["H1"].copy()
H2_check = states["H2"].copy()
errors_before = compute_errors(X, Y, parameters, H1_check, H2_check)
energy_before = compute_energy(Y, errors_before)

H1_check, H2_check, errors_after = inference(
    X, Y, parameters, H1_check, H2_check, steps=20, activity_rate=0.01
)
print("Energy before extra inference:", energy_before)
print("Energy after extra inference: ", compute_energy(Y, errors_after))

Energy before extra inference: 0.009569468611895303
Energy after extra inference:  0.007800198410500879


/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_1378/467788396.py:3: RuntimeWarning: divide by zero encountered in matmul
  mu2 = np.tanh(parameters["W2"] @ H1 + parameters["b2"])
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_1378/467788396.py:3: RuntimeWarning: overflow encountered in matmul
  mu2 = np.tanh(parameters["W2"] @ H1 + parameters["b2"])
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_1378/467788396.py:3: RuntimeWarning: invalid value encountered in matmul
  mu2 = np.tanh(parameters["W2"] @ H1 + parameters["b2"])
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_1378/467788396.py:4: RuntimeWarning: divide by zero encountered in matmul
  Z3 = parameters["W3"] @ H2 + parameters["b3"]
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_1378/467788396.py:4: RuntimeWarning: overflow encountered in matmul
  Z3 = parameters["W3"] @ H2 + parameters["b3"]
/var/folders/nx/p9q2w9hd2_d630_70vbtf_dc0000gn/T/ipykernel_1378/467788396.py:4